# ZS601 v3：L4 冒烟与完整 A/B 实验

固定代码、固定十相机；数据先从 Drive 复制到 Colab 本地。默认 A/B 各 150000 步，每1000步验证。新增图仅为1σ彩色实体椭球；保留RGB、法向、深度PNG；不保存验证NPZ。所有运行建立新目录，不覆盖已有结果。

请先选择 L4 GPU。完整训练消耗现有 Colab 额度。依次运行单元；任何报错均先检查，不使用“忽略错误继续”。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import sys,subprocess,uuid,json,shutil,zipfile,csv,os
import numpy as np
import torch
CODE_REF='d1df6c3e06f085abf2567b07eb78eef8a8ca1441'
REPO_URL='https://github.com/VISjudy/ZS601_3DGS.git'
DATA_ZIP=Path('/content/drive/MyDrive/LCCDataset/ZS601meetingroom/ZS601meetingroom_data.zip')
HISTORY=Path('/content/drive/MyDrive/LCCDataset/zs601_output/gaussian-splattingWithMask_v3_cff221ccfb')
VAL_SOURCE=HISTORY/'images-val10.txt'
TEST_SOURCE=HISTORY/'images_test.txt'
WORK=Path('/content')/('zs601_complete_'+uuid.uuid4().hex[:10]);WORK.mkdir()
RESULTS=HISTORY.parent/WORK.name;RESULTS.mkdir()
ITERATIONS=150000
COMMON={'sh_degree':2,'position_lr_init':0.000016,'position_lr_final':0.00000016,
        'position_lr_max_steps':150000,'scaling_lr':0.0015,'seed':42,'lazy_cache':100}
VAL_ELLIPSOIDS='on'
OVERRIDES={} # 单项覆盖示例 {'normal_loss':'off'}；默认保持A/B定义
def run(args,cwd=None):
    args=list(map(str,args));print('COMMAND',args,flush=True)
    logfile=RESULTS/('command_'+uuid.uuid4().hex[:10]+'.log')
    with logfile.open('x') as log,subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as p:
        for line in p.stdout:
            log.write(line)
            if not line.startswith('Reading camera'): print(line,end='',flush=True)
        if p.wait():raise subprocess.CalledProcessError(p.returncode,args)
    print('LOG',logfile)
assert torch.cuda.is_available() and 'L4' in torch.cuda.get_device_name(0)
run(['nvidia-smi'])
print({'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda})
run(['git','clone','--branch','v3-ab',REPO_URL,WORK/'repo'])
run(['git','checkout','--detach',CODE_REF],cwd=WORK/'repo')
CODE=WORK/'repo/gaussian-splattingWithMask_v3'


## 环境构建
保留 Colab 自带 PyTorch，源码已包含 `<cstdint>` 兼容修复。使用 CuPy CUDA 渲染椭球，不依赖 EGL；曾实测 EGL 落到 llvmpipe CPU。构建输出持续显示并保存日志。

In [ ]:
run([sys.executable,'-m','pip','install','ninja','plyfile','laspy','scipy','pillow','opencv-python-headless','tqdm','cupy-cuda12x'])
os.environ['MAX_JOBS']='2'
for name in ['simple-knn','diff-gaussian-rasterization']:
    run([sys.executable,'-m','pip','install','-v','--no-build-isolation',CODE/'submodules'/name])
run([sys.executable,'-c','import torch,cupy; from simple_knn._C import distCUDA2; from diff_gaussian_rasterization import GaussianRasterizer; print(torch.cuda.get_device_name(0),cupy.__version__)'],cwd=CODE)
run([sys.executable,'-m','unittest','-v','test_geometry_v3'],cwd=CODE)


## 输入与固定划分
固定val/test列表使用已恢复并验证的版本：10张val、135张test（val包含在test中），训练2559张且不重叠。文件缺失时停止，不重新随机抽样。输入txt优先级明确，不读取可能绕过划分的images.bin。


In [ ]:
for p in [DATA_ZIP,VAL_SOURCE,TEST_SOURCE]:assert p.is_file(),str(p)
LOCAL_ZIP=WORK/'data.zip';shutil.copy2(DATA_ZIP,LOCAL_ZIP)
DATA=WORK/'data';DATA.mkdir()
with zipfile.ZipFile(LOCAL_ZIP) as z:
    for item in z.infolist():
        target=(DATA/item.filename).resolve()
        assert target.is_relative_to(DATA.resolve()),item.filename
        assert (item.external_attr>>16)&0o170000!=0o120000,'ZIP symlink'
    z.extractall(DATA)
POINTS=DATA/'ZS601_3cm_sample.las'
INTR=DATA/'sparse/cameras.txt';POSES=DATA/'sparse/images.txt'
VAL=WORK/'images-val10.txt';TEST=WORK/'images_test.txt';TRAIN=WORK/'images_train_v3.txt'
shutil.copy2(VAL_SOURCE,VAL);shutil.copy2(TEST_SOURCE,TEST)
for p in [POINTS,INTR,POSES]:assert p.is_file(),str(p)
run([sys.executable,'prepare_v3.py','--images_file',POSES,'--val_file',VAL,'--test_file',TEST,'--output_train',TRAIN],cwd=CODE)
for p in [VAL,TEST,TRAIN]:shutil.copy2(p,RESULTS/p.name)
print('LOCAL INPUTS',DATA,POINTS,INTR,TRAIN,VAL,TEST)


In [ ]:
def train_command(group,out,iterations,resume=None,checkpoint_interval=50000):
    cmd=[sys.executable,'train_mask_v3.py','--experiment',group,'-s',DATA,'-m',out,
         '--point_cloud',POINTS,'--train_file',TRAIN,'--val_file',VAL,'--test_file',TEST,
         '--cameras_file',INTR,'--units','scene','--iterations',iterations,
         '--val_interval',1000,
         '--checkpoint_interval',checkpoint_interval,'--val_npz','off','--val_ellipsoids',VAL_ELLIPSOIDS]
    for key,value in {**COMMON,**OVERRIDES}.items():cmd+=['--'+key,str(value)]
    if resume:cmd+=['--resume',resume]
    return cmd
def verify(out,steps):
    done=json.loads((out/'completed.json').read_text());assert done['iteration']==steps
    loss=list(csv.DictReader((out/'loss_log.csv').open()))
    assert [int(r['iteration']) for r in loss]==list(range(1,steps+1))
    assert all(np.isfinite(float(r['total'])) for r in loss)
    val=list(csv.DictReader((out/'val_metrics.csv').open()))
    expected=[0]+[i for i in range(1,steps+1) if i%1000==0 or i==steps]
    assert len(val)==11*len(expected)
    for i in expected:
        d=out/'val_v3'/f'iteration_{i:06d}'
        for kind in ['rgb','normal','depth']+(['ellipsoid'] if VAL_ELLIPSOIDS=='on' else []):
            assert len(list(d.glob('*_'+kind+'.png')))==10,(d,kind)
        assert not list(d.glob('*_geometry.npz'))
    assert (out/'checkpoints'/f'iteration_{steps}.pth').is_file()
    print('VERIFIED',out,done,'loss rows',len(loss),'val rows',len(val))
    return {'path':str(out),**done,'loss_rows':len(loss),'val_rows':len(val)}


## 冒烟
先完成200步并验证CSV、固定十视角和checkpoint。短测试只验证工作流，不代表最终视觉质量。

In [ ]:
SMOKE={}
for group in ['A','B']:
    out=RESULTS/('smoke_'+group);run(train_command(group,out,200),cwd=CODE)
    SMOKE[group]=verify(out,200)


## 正式实验
A：法向/扁平初始化、相机定向、剪枝；B另外开启中心贴面、切向偏移、法向、厚度、切向尺寸五项约束。关闭增密、opacity reset和深度训练。每组从相同LiDAR初始化独立运行150000步。

共同优化参数参考用户第一版：SH=2，位置学习率1.6e-5降至1.6e-7，衰减周期150000步，尺度学习率0.0015，seed42，CPU懒加载缓存100。200步冒烟也保持150000步学习率周期。v3 A/B定义保持不变，不等同旧版标准初始化/扁平初始化对照；旧版增密参数不传入v3。每1000步输出验证PNG与CSV，每50000步保存完整checkpoint和PLY，因此同时保留50000/100000/150000步产物。

`loss_log.csv`：每步RGB L1、DSSIM、total、点数、累计耗时以及五项几何loss的raw/weight/weighted。
`val_metrics.csv`：每1000步10个相机及MEAN行，含有效像素PSNR/MAE，另列`ssim_zero_mask_full_image`明确表示置零mask后的整图SSIM，不能误称有效窗口SSIM。


In [ ]:
FORMAL={}
for group in ['A','B']:
    out=RESULTS/('formal_'+group);run(train_command(group,out,ITERATIONS),cwd=CODE)
    FORMAL[group]=verify(out,ITERATIONS)
with (RESULTS/'workflow_verified.json').open('x') as f:
    json.dump({'code_ref':CODE_REF,'smoke':SMOKE,'formal':FORMAL},f,indent=2)
print('A/B completed and verified:',RESULTS)


## 恢复与经验
- 只从完整checkpoint恢复，使用同一代码和配置，输出到新目录；原实验保留。恢复后CSV记录续跑区间，汇总时按iteration拼接，不能套用从1开始的verify。
- 历史100→200恢复可运行，但严格逐参数1e-5一致性未通过；差异小于独立重复训练的差异，不宣称位级确定性。
- 原始mask中0/1编码会触发加载器提示；既有冒烟的10张masked GT已与历史基线逐像素匹配。
- 1σ实体图保留真实厚度，固定opacity阈值0.05，使用DC颜色与方向光。薄圆盘和暗天花板不等同渲染失败。
- CuPy uint64哨兵必须显式使用numpy.uint64，避免Python整数转有符号类型溢出。
- 输入和重要输出在Drive中保留；关闭运行时前确认completed、CSV、图像、checkpoint和notebook均已保存。


In [ ]:
# 仅在全部产物保存且验证完成后手动设为True，释放GPU运行时。
DISCONNECT_WHEN_VERIFIED=False
if DISCONNECT_WHEN_VERIFIED:
    assert (RESULTS/'workflow_verified.json').is_file()
    from google.colab import runtime
    runtime.unassign()
